In [ ]:
import sys
sys.path.append('../')
import time

import numpy as np
import matplotlib.pyplot as plt

import epics
from siriuspy.devices import CAXCtrl
from caxscripts.h5file import HDF5File

In [ ]:
cax = CAXCtrl()

# Functions

Useful functions for the scan

In [ ]:
def open_all_slits():
    cax.slit_A1.move_robust_top(value=19.7-0.04)
    cax.slit_A1.move_robust_bottom(value=35.8)
    cax.slit_A1.move_robust_left(value=43.6-0.04)
    cax.slit_A1.move_robust_right(value=47.2)

# Scan

## Parameters

In [ ]:
pos_top_open = 19.7 - 0.04
pos_bottom_open = 35.8
pos_left_open = 43.6 - 0.04
pos_right_open = 47.2

rangex = 1.3 + 0.1
rangey = 4.8

proport = rangey/rangex

In [ ]:
L = 0.5

Ndxs = 7
dxs = np.linspace(0,rangex-L,Ndxs)
dys = np.linspace(0,rangey-L,int(proport*Ndxs))

# Execution

Initial state

In [ ]:
top0 = cax.slit_A1.top_pos
bottom0 = cax.slit_A1.bottom_pos
left0 = cax.slit_A1.left_pos
right0 = cax.slit_A1.right_pos

Initializate file

In [ ]:
filename = 'square_scan_.h5'

file = HDF5File(filename=filename) #,filedir=)

file.save_metadata(metadata_dict={
    'exposure_time': cax.dvf_B1.exposure_time,
    'slit_top': cax.slit_A1.top_pos,
    'slit_bottom': cax.slit_A1.bottom_pos,
    'slit_left': cax.slit_A1.left_pos,
    'slit_right': cax.slit_A1.right_pos
})

Loop

In [ ]:
t0 = time.time()

for i, dy in enumerate(dys):

    cax.slit_A1.move_robust_top(value=pos_top_open-rangey+L+dy)
    cax.slit_A1.move_robust_bottom(value=pos_bottom_open-dy)
    cax.slit_A1.move_robust_left(value=pos_left_open)
    cax.slit_A1.move_robust_right(value=pos_right_open-rangex+L)

    for j, dx in enumerate(dxs):

        print(f'scan step x {j+1}/{len(dxs)} and step y {i+1}/{len(dys)}')

        cax.slit_A1.move_robust_left(value=pos_left_open-dx)
        cax.slit_A1.move_robust_right(value=pos_right_open-rangex+L+dx)

        img1 = cax.dvf_A1.image
        img2 = cax.dvf_B1.image

        metadata = {
            'slit_top'    : cax.slit_A1.top_pos,
            'slit_bottom' : cax.slit_A1.bottom_pos,
            'slit_left'   : cax.slit_A1.left_pos,
            'slit_right'  : cax.slit_A1.right_pos
        }

        file.save_group(grpname=f'scan-{i}-{j}',groupmetadata=metadata)
        file.save_dataset(dsetname=f'dvf1-img',grpname=f'scan-{i}-{j}',dsetdata=img1)
        file.save_dataset(dsetname=f'dvf2-img',grpname=f'scan-{i}-{j}',dsetdata=img2)

open_all_slits()

dt = time.time()-t0
print(f'ellapsed time [s]: {dt}')

Back to initial state

In [ ]:
cax.slit_A1.move_robust_top(value=top0)
cax.slit_A1.move_robust_bottom(value=bottom0)
cax.slit_A1.move_robust_left(value=left0)
cax.slit_A1.move_robust_right(value=right0)